# Sistemas Inteligentes I — Baseline
## Búsqueda adversarial: poda Alfa–Beta

Versión limpia del notebook original para usar como punto de partida.

- **Alfa:** mejor valor asegurado para MAX.
- **Beta:** mejor valor asegurado para MIN.
- **Poda:** se corta una rama cuando alfa es mayor o igual que beta.
- **Correctitud:** el valor final coincide con Minimax.

> La lógica de los ejercicios originales se conserva. Se eliminaron celdas repetitivas de depuración y texto que no es necesario para modificar el taller.


# 1. Punto de partida: el problema de Minimax

Minimax supone que:

- MAX intenta maximizar;
- MIN intenta minimizar;
- ambos jugadores actúan racionalmente.

El problema es que, para garantizar su decisión, Minimax puede explorar aproximadamente:

$$O(b^m)$$

nodos.

Sin embargo, algunas ramas pueden resultar irrelevantes.

La idea central de Alfa–Beta es:

> **dejar de explorar una rama cuando ya sabemos que no puede mejorar la decisión de un jugador.**

La poda no cambia el resultado de Minimax.  
Solo intenta obtenerlo explorando menos nodos.

# 2. Recordatorio: un árbol de juego

Utilizaremos inicialmente el mismo tipo de representación:

```text
                    A  (MAX)
              /         |         \
          B (MIN)    C (MIN)    D (MIN)
          / | \       / | \       / | \
         3  5  2     9  1  4     6  7  8
```

Minimax calcula:

- `B = 2`
- `C = 1`
- `D = 6`

y finalmente:

$$A=\max(2,1,6)=6$$

In [1]:
arbol = {
    "A": ["B", "C", "D"],
    "B": ["B1", "B2", "B3"],
    "C": ["C1", "C2", "C3"],
    "D": ["D1", "D2", "D3"],
}

utilidades = {
    "B1": 3, "B2": 5, "B3": 2,
    "C1": 9, "C2": 1, "C3": 4,
    "D1": 6, "D2": 7, "D3": 8,
}

arbol, utilidades

({'A': ['B', 'C', 'D'],
  'B': ['B1', 'B2', 'B3'],
  'C': ['C1', 'C2', 'C3'],
  'D': ['D1', 'D2', 'D3']},
 {'B1': 3,
  'B2': 5,
  'B3': 2,
  'C1': 9,
  'C2': 1,
  'C3': 4,
  'D1': 6,
  'D2': 7,
  'D3': 8})

# 3. ¿Qué representan alfa y beta?

Durante la búsqueda mantenemos dos límites.

### Alfa — $\alpha$

Es el mejor valor que **MAX puede garantizar hasta el momento**.

Inicialmente:

$$\alpha=-\infty$$

### Beta — $\beta$

Es el mejor valor que **MIN puede garantizar hasta el momento**.

Inicialmente:

$$\beta=+\infty$$

Durante la búsqueda:

- MAX actualiza $\alpha$;
- MIN actualiza $\beta$.

Cuando ocurre:

$$\boxed{\alpha \geq \beta}$$

podemos realizar una **poda**.

# 4. Intuición de una poda

Suponga que MAX ya dispone de una alternativa con valor `6`.

Ahora explora otra rama cuyo turno pertenece a MIN.

Si MIN encuentra dentro de esa rama una opción con valor `4`, sabemos que podrá forzar:

$$valor \leq 4$$

MAX ya dispone de `6`, por lo que nunca elegirá una alternativa que termine en
`4` o menos.

Por tanto:

> **el resto de esa rama ya no puede cambiar la decisión de MAX.**

Podemos dejar de explorarla.

# 5. Implementación de Alfa–Beta

La estructura es muy similar a Minimax.

La diferencia está en que conservamos y actualizamos los límites
$\alpha$ y $\beta$.

In [2]:
from math import inf

def alfa_beta(nodo, es_max, arbol, utilidades, alfa=-inf, beta=inf):
    if nodo in utilidades:
        return utilidades[nodo]

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor = max(
                valor,
                alfa_beta(hijo, False, arbol, utilidades, alfa, beta)
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor = min(
                valor,
                alfa_beta(hijo, True, arbol, utilidades, alfa, beta)
            )

            beta = min(beta, valor)

            if alfa >= beta:
                break

        return valor


alfa_beta("A", True, arbol, utilidades)

6

# 6. Comparar Alfa–Beta con Minimax

Primero implementaremos una versión sencilla de Minimax.

In [3]:
def minimax(nodo, es_max, arbol, utilidades):
    if nodo in utilidades:
        return utilidades[nodo]

    valores = [
        minimax(hijo, not es_max, arbol, utilidades)
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


print("Minimax   :", minimax("A", True, arbol, utilidades))
print("Alfa-Beta :", alfa_beta("A", True, arbol, utilidades))

Minimax   : 6
Alfa-Beta : 6


# 7. Alfa–Beta paso a paso

Ahora imprimiremos:

- nodo visitado;
- jugador;
- valor de $\alpha$;
- valor de $\beta$;
- momento en el que se produce una poda.

In [4]:
def alfa_beta_debug(
    nodo,
    es_max,
    arbol,
    utilidades,
    alfa=-inf,
    beta=inf,
    profundidad=0
):
    sangria = "    " * profundidad
    jugador = "MAX" if es_max else "MIN"

    if nodo in utilidades:
        print(
            f"{sangria}{nodo}: terminal = {utilidades[nodo]} "
            f"[α={alfa}, β={beta}]"
        )
        return utilidades[nodo]

    print(
        f"{sangria}{nodo}: {jugador} "
        f"[α={alfa}, β={beta}]"
    )

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor_hijo = alfa_beta_debug(
                hijo,
                False,
                arbol,
                utilidades,
                alfa,
                beta,
                profundidad + 1
            )

            valor = max(valor, valor_hijo)
            alfa = max(alfa, valor)

            print(
                f"{sangria}  después de {hijo}: "
                f"valor={valor}, α={alfa}, β={beta}"
            )

            if alfa >= beta:
                print(
                    f"{sangria}  PODA en {nodo}: "
                    f"α={alfa} >= β={beta}"
                )
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor_hijo = alfa_beta_debug(
                hijo,
                True,
                arbol,
                utilidades,
                alfa,
                beta,
                profundidad + 1
            )

            valor = min(valor, valor_hijo)
            beta = min(beta, valor)

            print(
                f"{sangria}  después de {hijo}: "
                f"valor={valor}, α={alfa}, β={beta}"
            )

            if alfa >= beta:
                print(
                    f"{sangria}  PODA en {nodo}: "
                    f"α={alfa} >= β={beta}"
                )
                break

        return valor


alfa_beta_debug("A", True, arbol, utilidades)

A: MAX [α=-inf, β=inf]
    B: MIN [α=-inf, β=inf]
        B1: terminal = 3 [α=-inf, β=inf]
      después de B1: valor=3, α=-inf, β=3
        B2: terminal = 5 [α=-inf, β=3]
      después de B2: valor=3, α=-inf, β=3
        B3: terminal = 2 [α=-inf, β=3]
      después de B3: valor=2, α=-inf, β=2
  después de B: valor=2, α=2, β=inf
    C: MIN [α=2, β=inf]
        C1: terminal = 9 [α=2, β=inf]
      después de C1: valor=9, α=2, β=9
        C2: terminal = 1 [α=2, β=9]
      después de C2: valor=1, α=2, β=1
      PODA en C: α=2 >= β=1
  después de C: valor=2, α=2, β=inf
    D: MIN [α=2, β=inf]
        D1: terminal = 6 [α=2, β=inf]
      después de D1: valor=6, α=2, β=6
        D2: terminal = 7 [α=2, β=6]
      después de D2: valor=6, α=2, β=6
        D3: terminal = 8 [α=2, β=6]
      después de D3: valor=6, α=2, β=6
  después de D: valor=6, α=6, β=inf


6

# 8. Un ejemplo diseñado para observar podas

El orden de los valores del árbol anterior no siempre produce una poda muy visible.

Usaremos ahora este árbol:

```text
                         A (MAX)
                    /             \
               B (MIN)           C (MIN)
              /      \           /      \
          D(MAX)   E(MAX)    F(MAX)    G(MAX)
           3  5     6  9      1  2      0 -1
```

La exploración se realiza de izquierda a derecha.

In [5]:
arbol_poda = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G1", "G2"],
}

utilidades_poda = {
    "D1": 3, "D2": 5,
    "E1": 6, "E2": 9,
    "F1": 1, "F2": 2,
    "G1": 0, "G2": -1,
}

print("Minimax:", minimax("A", True, arbol_poda, utilidades_poda))
print()
alfa_beta_debug("A", True, arbol_poda, utilidades_poda)

Minimax: 5

A: MAX [α=-inf, β=inf]
    B: MIN [α=-inf, β=inf]
        D: MAX [α=-inf, β=inf]
            D1: terminal = 3 [α=-inf, β=inf]
          después de D1: valor=3, α=3, β=inf
            D2: terminal = 5 [α=3, β=inf]
          después de D2: valor=5, α=5, β=inf
      después de D: valor=5, α=-inf, β=5
        E: MAX [α=-inf, β=5]
            E1: terminal = 6 [α=-inf, β=5]
          después de E1: valor=6, α=6, β=5
          PODA en E: α=6 >= β=5
      después de E: valor=5, α=-inf, β=5
  después de B: valor=5, α=5, β=inf
    C: MIN [α=5, β=inf]
        F: MAX [α=5, β=inf]
            F1: terminal = 1 [α=5, β=inf]
          después de F1: valor=1, α=5, β=inf
            F2: terminal = 2 [α=5, β=inf]
          después de F2: valor=2, α=5, β=inf
      después de F: valor=2, α=5, β=2
      PODA en C: α=5 >= β=2
  después de C: valor=5, α=5, β=inf


5

# 9. Medir el ahorro de exploración

Para comparar los algoritmos contabilizaremos:

- nodos visitados;
- hojas evaluadas;
- podas realizadas.

In [6]:
def minimax_contando(nodo, es_max, arbol, utilidades, estadisticas):
    estadisticas["visitados"] += 1

    if nodo in utilidades:
        estadisticas["hojas"] += 1
        return utilidades[nodo]

    valores = [
        minimax_contando(
            hijo,
            not es_max,
            arbol,
            utilidades,
            estadisticas
        )
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


def alfa_beta_contando(
    nodo,
    es_max,
    arbol,
    utilidades,
    estadisticas,
    alfa=-inf,
    beta=inf
):
    estadisticas["visitados"] += 1

    if nodo in utilidades:
        estadisticas["hojas"] += 1
        return utilidades[nodo]

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor = max(
                valor,
                alfa_beta_contando(
                    hijo,
                    False,
                    arbol,
                    utilidades,
                    estadisticas,
                    alfa,
                    beta
                )
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                estadisticas["podas"] += 1
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor = min(
                valor,
                alfa_beta_contando(
                    hijo,
                    True,
                    arbol,
                    utilidades,
                    estadisticas,
                    alfa,
                    beta
                )
            )

            beta = min(beta, valor)

            if alfa >= beta:
                estadisticas["podas"] += 1
                break

        return valor


stats_minimax = {"visitados": 0, "hojas": 0}
stats_ab = {"visitados": 0, "hojas": 0, "podas": 0}

valor_mm = minimax_contando(
    "A", True, arbol_poda, utilidades_poda, stats_minimax
)

valor_ab = alfa_beta_contando(
    "A", True, arbol_poda, utilidades_poda, stats_ab
)

print("Valor Minimax:", valor_mm)
print("Valor Alfa-Beta:", valor_ab)
print()
print("Minimax:", stats_minimax)
print("Alfa-Beta:", stats_ab)

Valor Minimax: 5
Valor Alfa-Beta: 5

Minimax: {'visitados': 15, 'hojas': 8}
Alfa-Beta: {'visitados': 11, 'hojas': 5, 'podas': 2}


### Preguntas de análisis

1. ¿Ambos algoritmos producen el mismo valor?
2. ¿Cuántas hojas evita evaluar Alfa–Beta?
3. ¿Qué información permite justificar una poda?
4. ¿Podar significa que la rama sea necesariamente mala?
5. ¿Podría una rama podada contener valores muy altos o muy bajos?

### Respuestas

1. Sí. En el árbol de la sección 9 los dos devuelven 5.

2. Minimax evaluó 8 hojas y Alfa–Beta 5, así que evitó 3. También visitó 11 nodos contra 15, con 2 podas.

3. El par alfa y beta. Cuando alfa es mayor o igual que beta, el jugador actual ya tiene una opción mejor en otra rama.

4. No. Significa que esa línea no cambia la decisión con lo que ya se sabe. La rama puede tener un 9 y aun así cortarse.

5. Sí. Una rama podada puede guardar valores muy altos o muy bajos. No entran en la decisión porque el adversario no las permitiría, o porque MAX ya tiene algo mejor.


# 10. El orden de exploración importa

Alfa–Beta es especialmente eficaz cuando primero examinamos las jugadas más prometedoras.

En el mejor caso, su complejidad puede aproximarse a:

$$O(b^{m/2})$$

en lugar de:

$$O(b^m)$$

Esto significa que, con un buen ordenamiento, puede ser posible explorar
aproximadamente el doble de profundidad usando recursos comparables.

Sin embargo:

> **Alfa–Beta sigue siendo correcto independientemente del orden.  
> El orden afecta cuánto poda, no el valor final.**

## 10.1 Comparar dos órdenes del mismo árbol

Crearemos dos versiones:

- una con un orden favorable;
- otra con un orden menos favorable.

Los valores terminales son exactamente los mismos.

In [7]:
arbol_buen_orden = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D2", "D1"],
    "E": ["E2", "E1"],
    "F": ["F2", "F1"],
    "G": ["G1", "G2"],
}

arbol_mal_orden = {
    "A": ["C", "B"],
    "B": ["E", "D"],
    "C": ["G", "F"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G2", "G1"],
}

def medir_alfa_beta(arbol):
    stats = {"visitados": 0, "hojas": 0, "podas": 0}
    valor = alfa_beta_contando(
        "A",
        True,
        arbol,
        utilidades_poda,
        stats
    )
    return valor, stats


print("Orden 1:", medir_alfa_beta(arbol_buen_orden))
print("Orden 2:", medir_alfa_beta(arbol_mal_orden))

Orden 1: (5, {'visitados': 11, 'hojas': 5, 'podas': 2})
Orden 2: (5, {'visitados': 14, 'hojas': 7, 'podas': 1})


### Preguntas de análisis

1. ¿Cambió el valor final?
2. ¿Cambió el número de nodos visitados?
3. ¿Por qué conocer primero una buena jugada ayuda a podar?
4. ¿Cómo podría un programa real ordenar las jugadas antes de examinarlas?

### Respuestas

1. No. Los dos órdenes devuelven 5.

2. Sí. El orden favorable visitó 11 nodos, 5 hojas y 2 podas. El desfavorable visitó 14 nodos, 7 hojas y 1 poda.

3. Porque alfa o beta se aprietan antes y las ramas siguientes fallan el test más temprano.

4. Ordenar las jugadas con una heurística: capturas primero, la última jugada buena, o una búsqueda poco profunda previa.


# 11. Caso aplicado: juego de las piedras

Retomaremos el juego:

- hay una pila de piedras;
- cada jugador puede retirar `1`, `2` o `3`;
- quien retira la última piedra gana.

Compararemos Minimax y Alfa–Beta sobre el mismo juego.

In [8]:
MOVIMIENTOS = (1, 2, 3)

def movimientos_validos(piedras):
    return [m for m in MOVIMIENTOS if m <= piedras]


def minimax_piedras_contando(piedras, turno_max, stats):
    stats["visitados"] += 1

    if piedras == 0:
        stats["hojas"] += 1
        return -1 if turno_max else 1

    valores = [
        minimax_piedras_contando(
            piedras - retirar,
            not turno_max,
            stats
        )
        for retirar in movimientos_validos(piedras)
    ]

    return max(valores) if turno_max else min(valores)


def alfa_beta_piedras(
    piedras,
    turno_max,
    stats,
    alfa=-inf,
    beta=inf
):
    stats["visitados"] += 1

    if piedras == 0:
        stats["hojas"] += 1
        return -1 if turno_max else 1

    if turno_max:
        valor = -inf

        for retirar in movimientos_validos(piedras):
            valor = max(
                valor,
                alfa_beta_piedras(
                    piedras - retirar,
                    False,
                    stats,
                    alfa,
                    beta
                )
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                stats["podas"] += 1
                break

        return valor

    else:
        valor = inf

        for retirar in movimientos_validos(piedras):
            valor = min(
                valor,
                alfa_beta_piedras(
                    piedras - retirar,
                    True,
                    stats,
                    alfa,
                    beta
                )
            )

            beta = min(beta, valor)

            if alfa >= beta:
                stats["podas"] += 1
                break

        return valor

## 11.1 Comparación experimental

In [9]:
for piedras in [6, 8, 10, 12]:
    stats_mm = {"visitados": 0, "hojas": 0}
    stats_ab = {"visitados": 0, "hojas": 0, "podas": 0}

    valor_mm = minimax_piedras_contando(
        piedras, True, stats_mm
    )

    valor_ab = alfa_beta_piedras(
        piedras, True, stats_ab
    )

    print(f"\n{piedras} piedras")
    print("  Minimax   :", valor_mm, stats_mm)
    print("  Alfa-Beta :", valor_ab, stats_ab)


6 piedras
  Minimax   : 1 {'visitados': 52, 'hojas': 24}
  Alfa-Beta : 1 {'visitados': 45, 'hojas': 19, 'podas': 13}

8 piedras
  Minimax   : -1 {'visitados': 177, 'hojas': 81}
  Alfa-Beta : -1 {'visitados': 134, 'hojas': 57, 'podas': 46}

10 piedras
  Minimax   : 1 {'visitados': 600, 'hojas': 274}
  Alfa-Beta : 1 {'visitados': 329, 'hojas': 133, 'podas': 126}

12 piedras
  Minimax   : -1 {'visitados': 2031, 'hojas': 927}
  Alfa-Beta : -1 {'visitados': 987, 'hojas': 409, 'podas': 407}


# 12. Obtener la mejor jugada con Alfa–Beta

En una aplicación real necesitamos devolver una **acción**, no solo el valor.

In [10]:
def mejor_jugada_alfa_beta_piedras(piedras):
    mejor_valor = -inf
    mejor_movimiento = None
    alfa = -inf
    beta = inf

    for retirar in movimientos_validos(piedras):
        stats = {"visitados": 0, "hojas": 0, "podas": 0}

        valor = alfa_beta_piedras(
            piedras - retirar,
            False,
            stats,
            alfa,
            beta
        )

        if valor > mejor_valor:
            mejor_valor = valor
            mejor_movimiento = retirar

        alfa = max(alfa, mejor_valor)

    return mejor_movimiento, mejor_valor


for piedras in range(1, 11):
    movimiento, valor = mejor_jugada_alfa_beta_piedras(piedras)

    print(
        f"{piedras:2d} piedras -> "
        f"retirar {movimiento}, valor {valor}"
    )

 1 piedras -> retirar 1, valor 1
 2 piedras -> retirar 2, valor 1
 3 piedras -> retirar 3, valor 1
 4 piedras -> retirar 1, valor -1
 5 piedras -> retirar 1, valor 1
 6 piedras -> retirar 2, valor 1
 7 piedras -> retirar 3, valor 1
 8 piedras -> retirar 1, valor -1
 9 piedras -> retirar 1, valor 1
10 piedras -> retirar 2, valor 1


# 13. Taller

Implemente Alfa–Beta para **Tres en raya** y compare con Minimax en el mismo tablero.

Ver en [archivo](1-tictactoe.ipynb) con sus [resultados](../../results/alpha-beta-pruning/1-tictactoe.md)

### Uso de IA generativa

- herramienta utilizada: Cursor
- propósito de uso: organizar el baseline, redactar respuestas y enlazar los talleres
- partes de la solución en las que fue empleada: este notebook y el taller de tres en raya
